#
#
 
F
e
d
e
r
a
t
e
d
 
L
e
a
r
n
i
n
g
 
L
a
b
:
 
L
o
g
i
s
t
i
c
 
r
e
g
r
e
s
s
i
o
n
 
f
o
r
 
c
l
a
s
s
i
f
i
c
a
t
i
o
n
 
o
f
 
h
a
n
d
-
d
r
a
w
n
 
g
e
s
t
u
r
e
s



For this lab, you will train a logistic regression classifier to recognize hand-drawn gestures from the Quick Draw dataset. Then you will test it on real hand drawings submitted by your classmates.

The lab has four setups:
- **Setup 0:** Train on Quick Draw data, test on Quick Draw
- **Setup 1:** Same model, tested on 70 hand-drawn images from 7 classmates
- **Setup 2:** Federated Learning with just Student A (3 hand-drawn sheets)
- **Setup 3:** FL with both students (A: 3 sheets, B: 4 sheets)


#### Setup 0 — Train a classifier on Quick Draw data

First, we'll train a logistic regression on a subset of the Quick Draw dataset. 200 samples per class for your student slice.


In [ ]:
!pip install flwr scikit-learn matplotlib numpy Pillow nest-asyncio seaborn opencv-python-headless PyMuPDF --quiet


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import urllib.request
import warnings
from PIL import Image
import PIL.ImageOps
import cv2
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import log_loss
import flwr as fl

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

print('All packages imported.')


In [ ]:
STUDENT_ID = 'A'                # 'A' or 'B'
SERVER_ADDRESS = 'localhost:8090'

CLASSES = ['cat', 'dog', 'sun', 'clock', 'mountain',
           'tent', 'tree', 'bird', 'star', 'face']
nclasses = len(CLASSES)
SAMPLES = 200
SLICE_START = {'A': 0, 'B': 200}[STUDENT_ID]
SERVER_IP = SERVER_ADDRESS.split(':')[0]
HTTP_PORT = 8081

print(f'Student: {STUDENT_ID}  |  Server: {SERVER_ADDRESS}')


In [ ]:
# Load Quick Draw data for this student (200 samples per class)
BASE = 'https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/'
X, y = [], []
for i, c in enumerate(['cat','dog','sun','clock','mountain',
                        'tent','tree','bird','star','face']):
    fn = c + '.npy'
    if not os.path.exists(fn): urllib.request.urlretrieve(BASE + fn, fn)
    X.append(np.load(fn)[SLICE_START:SLICE_START+SAMPLES].copy().astype('f4')/255.)
    y.append(np.full(SAMPLES, i))
X = np.vstack(X); y = np.concatenate(y)
print(f'Loaded {len(X)} samples ({SAMPLES} per class)')


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=9,
                                     train_size=0.7, test_size=0.3)


In [ ]:
clf = LogisticRegression(penalty='l2',
                         max_iter=1000, solver='saga').fit(X_train, y_train)


In [ ]:
accuracy = clf.score(X_test, y_test)
print(accuracy)


In [ ]:
y_pred_qd = clf.predict(X_test)
cm = confusion_matrix(y_test, y_pred_qd)

fig, ax = plt.subplots(figsize=(10, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Setup 0 — Confusion Matrix (Quick Draw test set)')
plt.xticks(rotation=45, ha='right')
plt.show()


In [ ]:
misclassified_qd = np.where(y_test != y_pred_qd)[0]

plt.figure(figsize=(15, 6))
plt.suptitle('Misclassified Examples (Setup 0 — Quick Draw)', fontsize=16)

for i, idx in enumerate(misclassified_qd[:10]):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[idx].reshape(28, 28), cmap=plt.cm.gray)
    plt.title(f'True: {CLASSES[y_test[idx]]}\nPred: {CLASSES[y_pred_qd[idx]]}', color='red')
    plt.axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


#### Extract hand-drawn images from 7 folded sheets

Seven classmates each drew all 10 classes on a folded 3×4 grid. Upload the photos, detect the grid, and extract individual drawings.


In [ ]:
from google.colab import files
 
uploaded = files.upload()
 
for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

pdf_fn = list(uploaded.keys())[0]
print(f'PDF loaded: {pdf_fn}')


In [ ]:
# Extract pages from PDF and set rotation for each
import fitz

doc = fitz.open(pdf_fn)
page_imgs = []
print(f'Found {len(doc)} pages in the PDF')

n_pages = len(doc)
n_cols = 3
n_rows = (n_pages + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten() if n_pages > 1 else [axes]
for i in range(n_pages):
    pix = doc[i].get_pixmap()
    img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
    page_imgs.append(img)
    axes[i].imshow(np.asarray(img), cmap='gray')
    axes[i].set_title(f'Sheet {i+1}')
    axes[i].axis('off')
for j in range(n_pages, len(axes)):
    axes[j].axis('off')
plt.tight_layout()
plt.show()

rotations = [270, 180, 180, 180, 180, 180]  # hardcoded rotations
print(f'Loaded {len(page_imgs)} sheets')


In [ ]:
# Define class order in your folded grid (change if yours is different)
GRID_ORDER = ['cat','dog','sun','clock','mountain',
              'tent','tree','bird','star','face',
              'blank','initials']

# Enter initials for each sheet
sheet_initials = []
for i in range(len(page_imgs)):
    print(f'\n--- Sheet {i+1} ---')
    initials = input('  Enter person initials: ')
    sheet_initials.append(initials.strip().upper())

MARGIN = 0.05
X_hand, y_hand, persons, sheet_ids = [], [], [], []

for sheet_idx in range(len(page_imgs)):
    pil_img = page_imgs[sheet_idx]
    rot = rotations[sheet_idx]
    print(f'Processing sheet {sheet_idx+1} ({sheet_initials[sheet_idx]}, rotation {rot})...', end=' ')

    if rot:
        pil_img = pil_img.rotate(-rot, expand=True, fillcolor='white')

    img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    _, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        print('paper not found, skipping'); continue
    paper = max(contours, key=cv2.contourArea)

    peri = cv2.arcLength(paper, True)
    approx = cv2.approxPolyDP(paper, 0.02 * peri, True)
    if len(approx) == 4:
        src_pts = approx.reshape(4, 2).astype(np.float32)
    else:
        x, y, w, h = cv2.boundingRect(paper)
        src_pts = np.array([[x, y], [x+w, y], [x+w, y+h], [x, y+h]], dtype=np.float32)

    s = src_pts.sum(axis=1)
    tl = src_pts[np.argmin(s)]
    br = src_pts[np.argmax(s)]
    d = np.diff(src_pts, axis=1)
    tr = src_pts[np.argmin(d)]
    bl = src_pts[np.argmax(d)]
    src_ordered = np.array([tl, tr, br, bl], dtype=np.float32)

    dst_w = int(max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl)))
    dst_h = int(max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr)))
    dst_pts = np.array([[0, 0], [dst_w-1, 0], [dst_w-1, dst_h-1], [0, dst_h-1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(src_ordered, dst_pts)
    warped = cv2.warpPerspective(gray, M, (dst_w, dst_h))

    mh, mw = int(MARGIN * dst_h), int(MARGIN * dst_w)
    paper_gray = warped[mh:dst_h-mh, mw:dst_w-mw]

    cell_h = paper_gray.shape[0] // 4
    cell_w = paper_gray.shape[1] // 3

    for row in range(4):
        for col in range(3):
            label = GRID_ORDER[row * 3 + col]
            if label in ('blank', 'initials'):
                continue
            cell = paper_gray[row*cell_h:(row+1)*cell_h, col*cell_w:(col+1)*cell_w]
            cell_pil = Image.fromarray(cell).resize((28, 28), Image.LANCZOS)
            cell_arr = np.array(cell_pil, dtype=np.float32)
            cell_arr = (255 - cell_arr) / 255.0
            X_hand.append(cell_arr.reshape(784))
            y_hand.append(CLASSES.index(label))
            persons.append(sheet_initials[sheet_idx])
            sheet_ids.append(sheet_idx)
    print('10 cells')

X_hand = np.array(X_hand); y_hand = np.array(y_hand)
persons = np.array(persons); sheet_ids = np.array(sheet_ids)
print(f'\nExtracted {len(X_hand)} images total ({len(X_hand)//10} per class)')
print(f'People: {np.unique(persons)}')


In [ ]:
# Verify: warped sheets with 3x4 grid overlay
n_cols = 3
n_rows = (len(page_imgs) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten() if len(page_imgs) > 1 else [axes]
MARGIN = 0.05
print('Green grid = cell boundaries. Yellow labels = class names.')

for s in range(len(page_imgs)):
    pil_img = page_imgs[s]
    rot = rotations[s]
    if rot: pil_img = pil_img.rotate(-rot, expand=True, fillcolor='white')
    img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: continue
    paper = max(contours, key=cv2.contourArea)
    peri = cv2.arcLength(paper, True)
    approx = cv2.approxPolyDP(paper, 0.02 * peri, True)
    if len(approx) == 4:
        src_pts = approx.reshape(4, 2).astype(np.float32)
    else:
        x, y, w, h = cv2.boundingRect(paper)
        src_pts = np.array([[x, y], [x+w, y], [x+w, y+h], [x, y+h]], dtype=np.float32)
    s_sum = src_pts.sum(axis=1)
    tl = src_pts[np.argmin(s_sum)]
    br = src_pts[np.argmax(s_sum)]
    d = np.diff(src_pts, axis=1)
    tr = src_pts[np.argmin(d)]
    bl = src_pts[np.argmax(d)]
    src_ordered = np.array([tl, tr, br, bl], dtype=np.float32)
    dst_w = int(max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl)))
    dst_h = int(max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr)))
    dst_pts = np.array([[0, 0], [dst_w-1, 0], [dst_w-1, dst_h-1], [0, dst_h-1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(src_ordered, dst_pts)
    warped = cv2.warpPerspective(gray, M, (dst_w, dst_h))
    mh, mw = int(MARGIN * dst_h), int(MARGIN * dst_w)
    paper_gray = warped[mh:dst_h-mh, mw:dst_w-mw]

    axes[s].imshow(paper_gray, cmap='gray')
    ch, cw = paper_gray.shape[0] // 4, paper_gray.shape[1] // 3
    for row in range(4):
        for col in range(3):
            label = GRID_ORDER[row * 3 + col]
            axes[s].axhline(row * ch, color='lime', linewidth=1)
            axes[s].axvline(col * cw, color='lime', linewidth=1)
            axes[s].text(col * cw + cw // 2, row * ch + ch // 4,
                        label, color='yellow', ha='center', va='bottom', fontsize=8)
    axes[s].axhline(4 * ch, color='lime', linewidth=1)
    axes[s].axvline(3 * cw, color='lime', linewidth=1)
    axes[s].set_title(f'Sheet {s+1}: {sheet_initials[s]}')
    axes[s].axis('off')

for j in range(len(page_imgs), len(axes)):
    axes[j].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Verify extracted cells: each row = class, each column = person
print('Each row = one class. Each column = one person.')
fig, axes = plt.subplots(10, 7, figsize=(14, 20))
for c in range(10):
    idxs = np.where(y_hand == c)[0]
    for j, idx in enumerate(idxs[:7]):
        axes[c, j].imshow(X_hand[idx].reshape(28, 28), cmap='gray')
        axes[c, j].set_title(persons[idx], fontsize=8)
        axes[c, j].axis('off')
    axes[c, 0].set_ylabel(CLASSES[c], fontsize=14)
plt.tight_layout()
plt.show()


#### Setup 1 — Test your Quick Draw model on real hand drawings

Now test the Setup 0 model on the 70 hand-drawn images you just extracted. This shows how well a model trained on Quick Draw (computer-drawn strokes) generalizes to real hand drawings.


In [ ]:
setup1_pred = clf.predict(X_hand)
setup1_acc = np.mean(setup1_pred == y_hand)
print(f'Setup 1 accuracy on hand-drawn images: {setup1_acc:.1%}')


In [ ]:
cm = confusion_matrix(y_hand, setup1_pred)
fig, ax = plt.subplots(figsize=(10, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Setup 1 — Confusion Matrix (hand-drawn images)')
plt.xticks(rotation=45, ha='right')
plt.show()


In [ ]:
misclassified = np.where(y_hand != setup1_pred)[0]
print(f'Misclassified {len(misclassified)} / {len(y_hand)}')
plt.figure(figsize=(15, 8))
plt.suptitle('Misclassified Examples (Setup 1 — hand-drawn)', fontsize=16)
n_show = min(10, len(misclassified))
for i in range(n_show):
    idx = misclassified[i]
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_hand[idx].reshape(28, 28), cmap=plt.cm.gray)
    plt.title(f'True: {CLASSES[y_hand[idx]]} ({persons[idx]})\nPred: {CLASSES[setup1_pred[idx]]}',
              color='red', fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()


#### How the Grid Extraction Works

The scanned sheets are processed automatically:

1. **Perspective warp** — OpenCV finds the paper's 4 corners and warps it to a flat, front-facing rectangle (corrects angled photos)
2. **Margin crop** — 5% is cropped from each edge to skip fold gutters and paper borders
3. **4x3 grid division** — each sheet is split into 4 rows x 3 columns, matching the fold layout
4. **Preprocessing** — each cell is grayscaled, resized to 28x28, inverted (white strokes on black), and scaled to [0,1]
5. **Storage** — the images are stored in `X_hand` with labels in `y_hand` and person initials in `persons`

OpenCV contour detection `->` perspective warp `->` uniform grid slice `->` `Image.resize(28,28)` `->` `(255 - cell) / 255.0`


#### Why Setup 1 Has Low Accuracy

The model trained in Setup 0 learned patterns from **Quick Draw** — clean, perfectly centered, computer-generated strokes with consistent line width and minimal noise.

Real hand drawings are different: uneven pressure, varying stroke width, positioning differences, and background noise. The model has never seen these variations, so it performs at **chance level (about 10% for 10 classes)** — it is essentially guessing.

This is expected and intentional. It demonstrates that *a model trained on clean digital strokes does not generalize to messy hand drawings.* The confusion matrix and misclassified examples show what the model finds confusing.


#### How Federated Learning Fixes This

Federated Learning (Setups 2 and 3) retrains the model on **your actual hand-drawn images** instead of Quick Draw data:

- **Setup 2** — Student A participates with 3 hand-drawn sheets (30 images). The model adapts to As drawing style.
- **Setup 3** — Both students participate (A: 30 images, B: 40 images). More diverse data = better generalization.

Each FL round:
1. Server sends the current global model to clients
2. Each client trains for 1 epoch on their local hand-drawn data
3. Clients send updated weights back to the server
4. Server averages the weights (FedAvg) and saves the aggregated model

The server uses Flower's FedAvg strategy with SaveOnAggregateStrategy and serves model files over HTTP.


#### Federated Learning

Now you will run Federated Learning (FL) rounds using the hand-drawn images as training data instead of Quick Draw. Your raw data never leaves your computer.

- **Setup 2:** Student A participates with 3 hand-drawn sheets (30 images)
- **Setup 3:** Both students participate (A: 3 sheets, B: 4 sheets)


In [ ]:
class SketchClient(fl.client.NumPyClient):
    def __init__(self, X_train, y_train, X_val, y_val):
        self.X_train = X_train; self.y_train = y_train
        self.X_val = X_val; self.y_val = y_val
        self.model = LogisticRegression(
            penalty='l2', max_iter=1, warm_start=True, solver='saga',
        )
        dummy_X = np.zeros((nclasses * 2, 784))
        dummy_y = np.repeat(np.arange(nclasses), 2)
        self.model.fit(dummy_X, dummy_y)
    def get_parameters(self, config):
        return [self.model.coef_, self.model.intercept_]
    def fit(self, parameters, config):
        self.model.coef_ = parameters[0]
        self.model.intercept_ = parameters[1]
        self.model.fit(self.X_train, self.y_train)
        return [self.model.coef_, self.model.intercept_], len(self.X_train), {}
    def evaluate(self, parameters, config):
        self.model.coef_ = parameters[0]
        self.model.intercept_ = parameters[1]
        loss = float(log_loss(self.y_val, self.model.predict_proba(self.X_val)))
        acc = float(self.model.score(self.X_val, self.y_val))
        return loss, len(self.X_val), {'accuracy': acc}
print('SketchClient ready.')


#### Setup 2 — Federated Learning (Student A, 3 sheets)

Student A trains on 3 hand-drawn sheets via FL. After rounds complete, the aggregated model is tested on all 70 hand-drawn images.


In [ ]:
# Split: first 3 uploaded sheets = Student A, last 4 = Student B
# (A has 3 sheets x 10 classes = 30 images, B has 4 x 10 = 40)
X_fl_a = X_hand[sheet_ids < 3]
y_fl_a = y_hand[sheet_ids < 3]
X_fl_b = X_hand[sheet_ids >= 3]
y_fl_b = y_hand[sheet_ids >= 3]
print(f"Student A: {len(X_fl_a)} images (sheets {np.unique(sheet_ids[sheet_ids<3])})")
print(f"Student B: {len(X_fl_b)} images (sheets {np.unique(sheet_ids[sheet_ids>=3])})")


In [ ]:
print('Connecting to', SERVER_ADDRESS, 'for Setup 2 (A only, 3 sheets)...')
client = SketchClient(X_fl_a, y_fl_a, X_test, y_test)
fl.client.start_numpy_client(server_address=SERVER_ADDRESS, client=client)
fl_model_s2 = client.model
print('Setup 2 complete.')


In [ ]:
# Evaluate Setup 2 model on all 70 hand-drawn images
s2_pred = fl_model_s2.predict(X_hand)
s2_acc = np.mean(s2_pred == y_hand)
print(f'Setup 2 accuracy: {s2_acc:.1%}')

cm = confusion_matrix(y_hand, s2_pred)
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES).plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Setup 2 — Confusion Matrix (hand-drawn images)')
plt.xticks(rotation=45, ha='right')
plt.show()

mis = np.where(y_hand != s2_pred)[0]
print(f'Misclassified {len(mis)}/{len(y_hand)}')
plt.figure(figsize=(15, 8))
plt.suptitle('Setup 2 — Misclassified (hand-drawn)', fontsize=16)
for i, idx in enumerate(mis[:10]):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_hand[idx].reshape(28, 28), cmap=plt.cm.gray)
    plt.title(f'True: {CLASSES[y_hand[idx]]} ({persons[idx]})\nPred: {CLASSES[s2_pred[idx]]}',
              color='red', fontsize=9)
    plt.axis('off')
plt.tight_layout(); plt.show()


#### Setup 3 — Federated Learning (Both students, A: 3 + B: 4)

Both students participate. A trains on 3 sheets (30 images), B on 4 sheets (40 images). Server waits for both clients before each round.


In [ ]:
print('Connecting to', SERVER_ADDRESS, 'for Setup 3...')
my_sheets = X_fl_a if STUDENT_ID == 'A' else X_fl_b
my_labels = y_fl_a if STUDENT_ID == 'A' else y_fl_b
client = SketchClient(my_sheets, my_labels, X_test, y_test)
fl.client.start_numpy_client(server_address=SERVER_ADDRESS, client=client)
fl_model_s3 = client.model
print('Setup 3 complete.')


In [ ]:
# Evaluate Setup 3 model on all 70 hand-drawn images
s3_pred = fl_model_s3.predict(X_hand)
s3_acc = np.mean(s3_pred == y_hand)
print(f'Setup 3 accuracy: {s3_acc:.1%}')

cm = confusion_matrix(y_hand, s3_pred)
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES).plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Setup 3 — Confusion Matrix (hand-drawn images)')
plt.xticks(rotation=45, ha='right')
plt.show()

mis = np.where(y_hand != s3_pred)[0]
print(f'Misclassified {len(mis)}/{len(y_hand)}')
plt.figure(figsize=(15, 8))
plt.suptitle('Setup 3 — Misclassified (hand-drawn)', fontsize=16)
for i, idx in enumerate(mis[:10]):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_hand[idx].reshape(28, 28), cmap=plt.cm.gray)
    plt.title(f'True: {CLASSES[y_hand[idx]]} ({persons[idx]})\nPred: {CLASSES[s3_pred[idx]]}',
              color='red', fontsize=9)
    plt.axis('off')
plt.tight_layout(); plt.show()


#### Compare all models on the hand-drawn test set


In [ ]:
models = [
    ('Setup 0 (Quick Draw)', clf, setup1_acc),
    ('Global model', global_model, global_model.score(X_hand, y_hand)),
    ('Setup 2 (FL, A only)', fl_model_s2, s2_acc),
    ('Setup 3 (FL, both)', fl_model_s3, s3_acc),
]
print(f'{"Model":<30} {"Accuracy":<12}  Prediction breakdown by person')
print('-' * 80)
for name, m, acc in models:
    preds = m.predict(X_hand)
    line = f'{name:<30} {acc:<8.1%}  '
    for p in np.unique(persons):
        mask = persons == p
        p_acc = np.mean(preds[mask] == y_hand[mask])
        line += f'{p}:{p_acc:.0%} '
    print(line)


#### Done!

| Concept | What happened |
|---------|--------------|
| **Setup 0** | Trained on Quick Draw, tested on Quick Draw |
| **Setup 1** | Same model tested on real hand-drawn images |
| **Setup 2** | FL with Student A's hand-drawn data (3 sheets) |
| **Setup 3** | FL with both students (A: 3, B: 4 sheets) |
| **Privacy** | Raw data never left each student's computer |
